姓名：李晨辉　　学号：2026009793　　日期：2026-11-11

# 04｜最终测试集评估（冻结实验后再运行）

**重要：** 只有在模型结构、超参数和实验结论已基于训练集/验证集确定后，才运行本 Notebook。测试集用于最终独立评估，不应用于反复调参或选择模型。

本 Notebook 评估预先定义的 MLP 与 CNN 基线。若你改变了模型结构，必须同步修改对应模型定义，确保与保存的权重匹配。

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 1e-3
NUM_WORKERS = 0  # 对初学者和 Windows 环境更稳妥
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.ToTensor()
full_train_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=False, download=True, transform=transform
)
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [55000, 5000],
    generator=torch.Generator().manual_seed(SEED)
)

def make_loaders(seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, generator=generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders()
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
print(f"训练集: {len(train_dataset)} | 验证集: {len(val_dataset)} | 测试集: {len(test_dataset)}")
print("注意：这是最终评估 Notebook。请确认模型与超参数已冻结；本 Notebook 将读取测试集，结果不得再用于调参。")


Device: cpu
训练集: 55000 | 验证集: 5000 | 测试集: 10000
注意：这是最终评估 Notebook。请确认模型与超参数已冻结；本 Notebook 将读取测试集，结果不得再用于调参。


In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.network(x)

class CNN(nn.Module):
    def __init__(self, channels_1=32, channels_2=64, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.features = nn.Sequential(
            nn.Conv2d(1, channels_1, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channels_1, channels_2, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels_2 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


In [3]:
def evaluate_test(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total_count = 0.0, 0, 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            preds = logits.argmax(dim=1)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (preds == labels).sum().item()
            total_count += batch_size
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
    return {
        "test_loss": total_loss / total_count,
        "test_accuracy_pct": 100.0 * total_correct / total_count,
        "labels": all_labels,
        "predictions": all_preds,
    }

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
rows = []
all_predictions = {}

for model_name, model_class in [("mlp", MLP), ("cnn", CNN)]:
    checkpoint = RESULTS_DIR / f"{model_name}_baseline_state_dict.pt"
    if not checkpoint.exists():
        print(f"跳过 {model_name.upper()}：未找到 {checkpoint}。请先运行对应基线 Notebook。")
        continue
    model = model_class().to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    metrics = evaluate_test(model, test_loader)
    rows.append({
        "model": model_name.upper(),
        "test_loss": metrics["test_loss"],
        "test_accuracy_pct": metrics["test_accuracy_pct"],
    })
    all_predictions[model_name] = metrics
    print(f"{model_name.upper()} | test loss={metrics['test_loss']:.4f} | "
          f"test accuracy={metrics['test_accuracy_pct']:.2f}%")

if rows:
    final_df = pd.DataFrame(rows)
    final_df.to_csv(RESULTS_DIR / "final_test_metrics.csv", index=False)
    display(final_df)
    print("测试集结果已保存。不要根据这次测试结果继续选择超参数。")


MLP | test loss=0.3724 | test accuracy=86.27%


CNN | test loss=0.2617 | test accuracy=90.75%


,model,test_loss,test_accuracy_pct
0,MLP,0.372386,86.27
1,CNN,0.261749,90.75


测试集结果已保存。不要根据这次测试结果继续选择超参数。


## 可选：查看错误案例

如需展示错误案例，请在报告中注明模型、样本编号、真实类别、预测类别及分析。不要把错误案例的主观解释当成确定的因果证据。

## 最终测试结果

设置冻结之后在测试集上跑了一次：

| 模型 | 测试损失 | 测试准确率 |
|---|---:|---:|
| MLP | 0.3724 | 86.27% |
| CNN | 0.2617 | 90.75% |

和验证集比，MLP 是 86.26%→86.27%，CNN 是 90.88%→90.75%，差得很少，说明没有对着测试集调参。这次结果只写进报告，没有回头改任何设置。
